In [1]:
%load_ext autoreload
%autoreload 2

import requests, os, zipfile, re
import pandas as pd, numpy as np
import peartree as pt, networkx as nx
from numba import jit
from dotenv import load_dotenv

In [2]:
# load api keys
load_dotenv()

API_KEY_ORS = os.getenv("OPENROUTESERVICE")

# KL GTFS data

In [3]:
# Get data from government API
def fetch_gtfs_data(category):

    # Download the GTFS ZIP file from the API
    url = f"https://api.data.gov.my/gtfs-static/prasarana?category={category}"
    response = requests.get(url)

    if response.status_code != 200:
        print(response.status_code)
        raise Exception(f"Failed to fetch GTFS data for category '{category}'")
    
    # Save the content of the response as a zip file
    file_name = f'gtfs_{category}.zip'
    with open(file_name, 'wb') as f:
        f.write(response.content)
        print(f"GTFS data saved as {file_name}.")

    
def gtfs_zip_to_dfs(category):

    file_name = f'gtfs_{category}.zip'
    gtfs_data = {}

    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        files = [f for f in zip_ref.namelist() if f.endswith('.txt')]
        
        for file in files:
            with zip_ref.open(file) as f:
                gtfs_data[file.replace('.txt', '')] = pd.read_csv(f, encoding='utf-8')

    print('GTFS files loaded: ', ', '.join([f.replace('.txt','') for f in files]))

    return gtfs_data

fetch_gtfs_data('rapid-rail-kl')

GTFS data saved as gtfs_rapid-rail-kl.zip.


In [4]:
# Load into dfs
gtfs = gtfs_zip_to_dfs('rapid-rail-kl')

# Access the data
stops = gtfs.get('stops')
routes = gtfs.get('routes')
trips = gtfs.get('trips')
freqs = gtfs.get('frequencies')
stti = gtfs.get('stop_times')
# shapes = gtfs.get('shapes')

GTFS files loaded:  agency, calendar, frequencies, routes, shapes, stop_times, stops, trips


In [14]:
nodes = stops[['stop_id','stop_name','stop_lat','stop_lon','route_id']].copy()
nodes = pd.merge(nodes, routes[['route_id','route_color','route_text_color']])
nodes.head()

,stop_id,stop_name,stop_lat,stop_lon,route_id,route_color,route_text_color
0,AG18,AMPANG,3.150318,101.760049,AG,e57200,FFFFFF
1,AG17,CAHAYA,3.140575,101.756677,AG,e57200,FFFFFF
2,AG16,CEMPAKA,3.138324,101.752979,AG,e57200,FFFFFF
3,AG15,PANDAN INDAH,3.134581,101.746509,AG,e57200,FFFFFF
4,AG14,PANDAN JAYA,3.130141,101.739122,AG,e57200,FFFFFF


In [28]:
freqs['line'] = freqs['trip_id'].str.split('_').str[0]
freqs['dow'] = freqs['trip_id'].str.split('_').str[1]
headway = freqs.groupby(['line','dow'])['headway_secs'].max().unstack()
headway.T

line,AGL,BRT,KGL,KJL,MRL,PYL,SPL
dow,,,,,,,
MonFri,300,480,600,420,600,840,300
Sat,300,360,600,420,720,600,300
Sun,300,360,600,420,720,600,300


## Peartree experimentation

Problems:
1. not clear how start_time and end_time affect the computation of stop_cost
2. using deprecated .iteritems which causes problems with newer versions of pandas

In [ ]:
G = pt.load_feed_as_graph(
	pt.get_representative_feed(f'gtfs_rapid-rail-kl.zip'),
	start_time = 7 * 3600, end_time = 15 * 3600,
	name = 'RAPID',  # node name prefixes
	existing_graph = None,
	connection_threshold = 50.0,  # max metres for 2 stops to be considered connected
    walk_speed_kmph = 4.5,
	stop_cost_method = lambda x, y, z: z,
    fallback_stop_cost = 5 * 60,  # 5 mins waiting time default
    interpolate_times = True,
    impute_walk_transfers = True,
    use_multiprocessing = False,
)

# Experimenting with routing

Getting the starting point for the pedestrian.
Heuristics:
1. Should be within 1km straight line distance from original location.
2. For each transit line, only pick the closest 1 station

In [15]:
# store station locations as tuples
stations = [(stop_id, stop_lat, stop_lon) for stop_id, stop_lat, stop_lon in zip(stops['stop_id'], stops['stop_lat'], stops['stop_lon'])]

In [16]:
# haversine formula
@jit(nopython=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # earth radius (km)

    # convert latlong from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1  # N-S distance
    dlon = lon2 - lon1  # E-W distance

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    distance = int(round(1000 * R * c))   # distance in metres, rounded to nearest metre

    return distance

In [17]:
orig = (3.158378738740368, 101.7084276959602)  # Pelita KLCC
dest = (3.1278264694956586, 101.72497173620582)  # Sunway Velocity

driver_orig = (3.1205620564395713, 101.70879519234487)  # Porsche Sungai Besi

In [18]:
# calculate distances between POI and all stations
def calculate_distances_poi_to_stations(ref_point, stations):
    return {
        station[0]: haversine(ref_point[0], ref_point[1], station[1], station[2])
        for station in stations
    }

In [19]:
orig_dist = calculate_distances_poi_to_stations(orig, stations)
# dest_dist = calculate_distances_poi_to_stations(dest, stations)

In [27]:
# filter stations within distance threshold, rank by proximity, 
# and limit to N stations per transit line
def filter_and_rank_stations(distance_tuples, threshold_metres=1000, max_stations_per_transit_line=1):

    # filter stations within the threshold distance
    filtered = {key: value for key, value in distance_tuples.items() if value <= threshold_metres}
    
    # rank the filtered stations by distance, nearest first
    ranked = dict(sorted(filtered.items(), key=lambda item: item[1]))
    
    # deduplicate: keep up to N stations for each transit line
    seen_prefixes = {}
    deduplicated = {}
    
    for key, value in ranked.items():

        # extract the transit line (alphabetic prefix of station name)
        match = re.match(r'^[a-zA-Z]+', key)
        if match:
            prefix = match.group(0)
            
            # initialize the transit line if it hasn't been seen yet
            if prefix not in seen_prefixes:
                seen_prefixes[prefix] = 0
            
            # if the transit line has fewer than M stations so far, keep the station
            if seen_prefixes[prefix] < max_stations_per_transit_line:
                deduplicated[key] = value
                seen_prefixes[prefix] += 1

    return deduplicated


In [30]:
orig_stations = filter_and_rank_stations(orig_dist, 1000, 2)
orig_stations

{'MR8': 468, 'MR7': 863}

Get the worst case scenario for the driver
- Driving to the passenger's origin, then to the destination.

In [58]:
def driving_route_btwn_multiple_points(*points):
    
    # ensure all the points are tuples with two elements (latitude, longitude)
    if not all(isinstance(point, tuple) and len(point) == 2 for point in points):
        raise ValueError("Each point must be a tuple with two elements: (latitude, longitude)")

    # reverse the order of the coordinates to (long, lat) as required by ORS
    points = [[point[1], point[0]] for point in points]  # Reverse the order (longitude, latitude)

    body = {
        'coordinates': points
    }

    headers = {
        'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
        'Authorization': API_KEY_ORS,
        'Content-Type': 'application/json; charset=utf-8'
    }

    # Make the request to the API
    resp = requests.post(
        'https://api.openrouteservice.org/v2/directions/driving-car', 
        json=body, 
        headers=headers
    )

    # Check for successful response
    if resp.status_code != 200:
        try:
            payload = resp.json()  # Try to parse the error response as JSON
            error_code = payload.get('error', {}).get('code', 'Unknown code')
            error_message = payload.get('error', {}).get('message', 'Unknown error message')
        except ValueError:
            error_code = 'Unknown code'
            error_message = 'Unknown error message'

        raise Exception(f"HTTP {resp.status_code}: {resp.reason}. Error code: {error_code}. Error message: {error_message}")

    # Parse the response payload
    try:
        payload = resp.json()
    except ValueError:
        raise ValueError(f"Failed to parse JSON response from OpenRouteService API: {resp.text}")

    return payload

# # DO NOT USE THIS - USE THE MULTI POINT VERSION ABOVE
# def driving_route_btwn_two_points(start, end):
#     headers = {
#         'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
#     }

#     # Make the request to the API
#     resp = requests.get(
#         f'https://api.openrouteservice.org/v2/directions/driving-car?api_key={API_KEY_ORS}&start={start[1]},{start[0]}&end={end[1]},{end[0]}', 
#         headers=headers
#     )

#     # if not 200, raise an error
#     if resp.status_code != 200:
#         try:
#             payload = resp.json()  # try to parse the error response as JSON
#             error_code = payload.get('error', {}).get('code', 'Unknown code')
#             error_message = payload.get('error', {}).get('message', 'Unknown error message')
#         except ValueError:
#             error_code = 'Unknown code'
#             error_message = 'Unknown error message'

#         raise Exception(f"HTTP {resp.status_code}: {resp.reason}. Error code: {error_code}. Error message: {error_message}")

#     # parse the payload
#     try:
#         payload = resp.json()
#     except ValueError:
#         raise ValueError(f"Failed to parse JSON response from OpenRouteService API: {resp.text}")
    
#     # todo - process response
#     return payload

In [ ]:
# process route to get total distance in metres and time in seconds
def estimate_driving_distance_and_duration(route):

	distance_duration_pairs = [segment['summary'] for segment in route['routes']]

	# if False:  # Backup for driving_route_btwn_two_points route
	# 	distance_duration_pairs = [segment['properties']['summary'] for segment in route['features']]
	
	total_distance_duration = {key: sum(pair[key] for pair in distance_duration_pairs) for key in ['distance', 'duration']}

	return total_distance_duration

In [ ]:
# Worst case scenario
wcs = estimate_driving_distance_and_duration(driving_route_btwn_multiple_points(driver_orig, orig, dest))
wcs

{'distance': 12114.2, 'duration': 1369.7}